# 02 — SVM v1 (eight classes)

The paper lists **nine** species. *Thalassia hemprichii* is held out until its folder is audited, so this run trains the **eight** trainable classes from the **`all`** folders only.

Skipped: research-grade, any `- Copy` folder, *Halophila spinulosa*, *Thalassia*.

**CPU is enough.** On this machine Run All reads `assets/species/Seagrass` and never mounts Google Drive. In Colab the same notebook mounts Drive and reads the `Seagrass` shortcut.

The pipeline is `ml/src`, the same code the identify server runs on a user photo. This notebook calls it, prints progress, and draws the charts.

Order: list → cap → 70/15/15 split → augment short train classes → features → **tune kernel, C, gamma on validation** → fit the chosen SVM → score train, validation, and test once → save a run record.

`MAX_PER_CLASS = 400` caps the large species. Use `100` for a quick smoke check. The local folders are short of the Drive set for *H. pinifolia* and *H. ovalis*, so a local score is not the paper score.

## A. Runtime and code

On this machine this cell skips Drive. In Colab it mounts Drive (same account as the `Seagrass` shortcut) and looks for the repo at `/content/marinesee` or `MyDrive/marinesee`. Clone or copy the repo there first.

In [ ]:
from datetime import datetime, timezone
from pathlib import Path
import json
import sys
import time


def running_in_colab() -> bool:
    try:
        import google.colab  # type: ignore
        return True
    except ImportError:
        return False


IN_COLAB = running_in_colab()
RUN_STARTED = datetime.now(timezone.utc)
print("runtime", "colab" if IN_COLAB else "local machine")
print("run started", RUN_STARTED.strftime("%Y-%m-%dT%H:%M:%SZ"))

if IN_COLAB:
    if not (Path("/content/drive/MyDrive").exists() or Path("/content/drive/My Drive").exists()):
        from google.colab import drive
        drive.mount("/content/drive", force_remount=True)
    print("drive ok")
else:
    print("Colab Drive mount skipped. Training reads photos on this machine.")


def repo_root() -> Path:
    seeds = [Path.cwd().resolve()]
    if IN_COLAB:
        seeds += [Path("/content/marinesee"), Path("/content/drive/MyDrive/marinesee")]
    for seed in seeds:
        for parent in [seed, *seed.parents]:
            for candidate in (parent, parent / "marinesee"):
                if (candidate / "ml" / "src" / "train_svm.py").is_file():
                    return candidate
    raise FileNotFoundError(
        "MarineSee repo not found. Locally: open this notebook from the marinesee folder "
        "with the MarineSee (.venv) kernel. In Colab: clone or copy the repo to /content/marinesee."
    )


REPO = repo_root()
if str(REPO / "ml") not in sys.path:
    sys.path.insert(0, str(REPO / "ml"))
print("repo", REPO)

In [ ]:
from collections import Counter

import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import classification_report
from tqdm.auto import tqdm

from src.dataset_contract import (
    HELD_OUT_SPECIES,
    IMAGE_SIZE,
    SPLIT_SEED,
    TRAIN_VARIANT,
    TRAINABLE_SPECIES,
)
from src.features import FEATURE_SPEC, extract_vector
from src.preprocess import cap_per_class, expand_minority_train, list_labeled_images, load_prepared
from src.run_log import append_run
from src.split import split_labels
from src.train_svm import evaluate, export_artifacts, make_pipeline
from src.tune_svm import RBF_C, RBF_GAMMA, TUNING_SPEC, best, best_params, candidate_params, search


def step(title: str) -> None:
    print(f"\n>>> {title}", flush=True)


MARKER = "cymodocea rotundata"


def looks_like_dataset_root(path: Path) -> bool:
    if not path.is_dir():
        return False
    names = " ".join(p.name.lower() for p in path.iterdir() if p.is_dir())
    return MARKER in names


def find_root() -> Path:
    if not IN_COLAB:
        local = REPO / "assets" / "species" / "Seagrass"
        if not looks_like_dataset_root(local):
            raise FileNotFoundError(f"Local dataset not found: {local}")
        return local
    for cand in (Path("/content/drive/MyDrive/Seagrass"), Path("/content/drive/My Drive/Seagrass")):
        if looks_like_dataset_root(cand):
            return cand
    for base in (Path("/content/drive/MyDrive"), Path("/content/drive/My Drive")):
        if not base.is_dir():
            continue
        for path in base.rglob("*"):
            if path.is_dir() and looks_like_dataset_root(path):
                return path
    raise FileNotFoundError("Seagrass dataset not found under My Drive")


DATA_ROOT = find_root()
print("DATA_ROOT", DATA_ROOT)
print("source", "google drive" if IN_COLAB else "this machine")
print("trainable", len(TRAINABLE_SPECIES), TRAINABLE_SPECIES)
print("features", FEATURE_SPEC)

## B. List and cap

The cap keeps *H. ovalis* and *Enhalus* from drowning the small species. Small species keep every photo they have.

In [ ]:
MAX_PER_CLASS = 400  # 100 = quick smoke check; None = every photo in DATA_ROOT

step("Listing images")
t0 = time.perf_counter()
all_pairs = list_labeled_images(DATA_ROOT)
print(f"listed {len(all_pairs)} in {time.perf_counter() - t0:.1f}s")
print("available", dict(Counter(slug for _, slug in all_pairs)))

pairs = cap_per_class(all_pairs, MAX_PER_CLASS, seed=SPLIT_SEED)
print("images this run", len(pairs), "max_per_class", MAX_PER_CLASS)
print("used", dict(Counter(slug for _, slug in pairs)))
assert {slug for _, slug in pairs} == set(TRAINABLE_SPECIES), "a trainable class is empty"

## C. Split, load, augment train

70 / 15 / 15 stratified, seed 42. Each photo is cropped to the plant when the green mask finds a mid-size plant region, then resized to 256. The identify server does the same crop on a user photo. Only train classes under 80 images get flip / rotate / zoom copies. Validation and test stay real photos.

In [ ]:
train_p, val_p, test_p, train_y, val_y, test_y = split_labels(
    [p for p, _ in pairs], [s for _, s in pairs], seed=SPLIT_SEED
)
print("split", len(train_p), len(val_p), len(test_p))


def load_many(paths, labels, tag):
    images, keep = [], []
    for path, label in tqdm(list(zip(paths, labels)), desc=f"Load {tag}", unit="img"):
        try:
            images.append(load_prepared(path, size=IMAGE_SIZE))
            keep.append(label)
        except Exception as exc:
            tqdm.write(f"skip {path}: {exc}")
    return images, keep


step("Loading images")
train_img, train_y = load_many(train_p, train_y, "train")
val_img, val_y = load_many(val_p, val_y, "val")
test_img, test_y = load_many(test_p, test_y, "test")
real_train_counts = dict(Counter(train_y))

step("Augment short train classes")
train_img, train_y = expand_minority_train(train_img, train_y, min_per_class=80)
print("train real ", real_train_counts)
print("train after", dict(Counter(train_y)))

## D. Features

`ml/src/features.py` v2, 148 numbers per photo. The identify server calls the same function on a user photo. The saved model uses the locked settings from the validation search: RBF `C=3`, `gamma=0.01`.

- **Color:** HSV histogram of the whole frame, HSV histogram of the plant pixels only, and the plant pixels' HSV mean and spread.
- **Texture:** GLCM on a 128px gray copy at distances 1, 2, 4. Six properties, each averaged over four angles plus its spread across angles.
- **Shape:** blobs in the green plant mask (Otsu fallback for brown blades), including blade width from the mask skeleton.

In [ ]:
def featurize(images, tag):
    t0 = time.perf_counter()
    rows = [extract_vector(im) for im in tqdm(images, desc=f"Features {tag}", unit="img")]
    print(f"features {tag} in {time.perf_counter() - t0:.1f}s")
    return np.stack(rows, axis=0)


step("Extracting features")
X_train = featurize(train_img, "train")
X_val = featurize(val_img, "val")
X_test = featurize(test_img, "test")
print(X_train.shape, X_val.shape, X_test.shape)

## E. Tune on validation

Each candidate SVM is fit on train and scored on validation. The test split is not used here. The pick is the highest **validation macro F1**, so *H. minor* counts as much as *H. ovalis*. Grid: RBF kernel over C × gamma, plus a linear kernel over C, all with `class_weight="balanced"`.

An SVM has no epoch loss curve. The line chart is the SVM equivalent: train against validation macro F1 as C grows, at the chosen gamma. A wide gap between the two lines means the model is memorizing training photos.

In [ ]:
step(f"Tuning {len(candidate_params())} SVM settings on validation")
tuning_rows = search(
    X_train,
    train_y,
    X_val,
    val_y,
    seed=SPLIT_SEED,
    progress=lambda grid: tqdm(grid, desc="Tune", unit="fit"),
)
chosen = best(tuning_rows)
BEST = best_params(tuning_rows)

ranked = sorted(tuning_rows, key=lambda r: r["val_macro_f1"], reverse=True)
print(f"{'kernel':<7} {'C':>7} {'gamma':>7} {'train F1':>9} {'val F1':>7} {'val acc':>8} {'gap':>6}")
for row in ranked[:10]:
    print(
        f"{row['kernel']:<7} {row['C']:>7} {str(row['gamma']):>7} "
        f"{row['train_macro_f1']:>9.3f} {row['val_macro_f1']:>7.3f} "
        f"{row['val_accuracy']:>8.3f} {row['f1_gap']:>6.3f}"
    )
print("\nchosen", BEST)

old_setting = next(
    (r for r in tuning_rows if r["kernel"] == "rbf" and r["C"] == 10.0 and r["gamma"] == "scale"),
    None,
)
if old_setting:
    print(
        "old fixed setting (rbf, C=10, gamma=scale):"
        f" val macro F1 {old_setting['val_macro_f1']:.3f}  gap {old_setting['f1_gap']:.3f}"
    )


def tuning_curve():
    rows = sorted(
        (r for r in tuning_rows if r["kernel"] == chosen["kernel"] and r["gamma"] == chosen["gamma"]),
        key=lambda r: r["C"],
    )
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.plot([r["C"] for r in rows], [r["train_macro_f1"] for r in rows], marker="o", label="train")
    ax.plot([r["C"] for r in rows], [r["val_macro_f1"] for r in rows], marker="o", label="validation")
    ax.axvline(chosen["C"], linestyle="--", color="gray")
    ax.set_xscale("log")
    ax.set_ylim(0, 1)
    ax.set_xlabel("C")
    ax.set_ylabel("macro F1")
    ax.set_title(f"{chosen['kernel']} gamma={chosen['gamma']}: train vs validation")
    ax.legend()
    fig.tight_layout()
    return fig


def tuning_heatmap():
    grid = np.zeros((len(RBF_GAMMA), len(RBF_C)))
    for r in tuning_rows:
        if r["kernel"] == "rbf":
            grid[RBF_GAMMA.index(r["gamma"]), RBF_C.index(r["C"])] = r["val_macro_f1"]
    fig, ax = plt.subplots(figsize=(7, 4))
    im = ax.imshow(grid, vmin=0, vmax=max(0.01, float(grid.max())))
    ax.set_xticks(range(len(RBF_C)))
    ax.set_xticklabels([str(c) for c in RBF_C])
    ax.set_yticks(range(len(RBF_GAMMA)))
    ax.set_yticklabels([str(g) for g in RBF_GAMMA])
    ax.set_xlabel("C")
    ax.set_ylabel("gamma")
    ax.set_title("RBF validation macro F1")
    for i in range(grid.shape[0]):
        for j in range(grid.shape[1]):
            ax.text(j, i, f"{grid[i, j]:.2f}", ha="center", va="center", fontsize=7, color="white")
    fig.colorbar(im, ax=ax, fraction=0.046)
    fig.tight_layout()
    return fig


curve_fig = tuning_curve()
plt.show()
heatmap_fig = tuning_heatmap()
plt.show()

In [ ]:
step(f"Fitting chosen SVM {BEST}")
model = make_pipeline(SPLIT_SEED, probability=True, **BEST)
t0 = time.perf_counter()
model.fit(X_train, train_y)
fit_seconds = time.perf_counter() - t0
print(f"fitted in {fit_seconds:.1f}s")

results = {}
for name, X, y in (("train", X_train, train_y), ("val", X_val, val_y), ("test", X_test, test_y)):
    results[name] = evaluate(model, X, y)
    macro = results[name]["report"]["macro avg"]["f1-score"]
    print(f"\n{name} accuracy {results[name]['accuracy']:.4f}  macro F1 {macro:.4f}")
    print(classification_report(y, results[name]["pred"], labels=list(TRAINABLE_SPECIES), zero_division=0))

test_acc = results["test"]["accuracy"]
cm = np.array(results["test"]["confusion_matrix"])
print("study target is 85% test accuracy. this run:", round(test_acc * 100, 2), "%")

## F. Save the run record

Every run appends `ml/artifacts/run_log.md`; that file is the history. This run's folder `ml/artifacts/runs/<id>-svm/` holds `record.md`, `scores.csv` (train, validation, test F1 per species), `tuning.csv` (every grid setting), the tuning curve, the tuning heatmap, the per-class F1 chart, and the confusion matrix.

The model goes to `ml/artifacts/v1/` on this machine, or Drive `Seagrass-ml-artifacts/v1/` in Colab.

In [ ]:
def save_f1_chart(path):
    labels = list(TRAINABLE_SPECIES)
    x = np.arange(len(labels))
    width = 0.25
    fig, ax = plt.subplots(figsize=(10, 4))
    for offset, name in ((-width, "train"), (0.0, "val"), (width, "test")):
        report = results[name]["report"]
        ax.bar(x + offset, [float(report[slug]["f1-score"]) for slug in labels], width, label=name)
    ax.set_xticks(x)
    ax.set_xticklabels(labels, rotation=75, ha="right", fontsize=8)
    ax.set_ylim(0, 1)
    ax.set_ylabel("F1")
    ax.set_title(f"per-class F1  test acc={test_acc:.3f}")
    ax.legend()
    fig.tight_layout()
    fig.savefig(path, dpi=120)
    plt.show()


def save_confusion(path):
    fig, ax = plt.subplots(figsize=(8, 7))
    im = ax.imshow(cm)
    ax.set_xticks(range(len(TRAINABLE_SPECIES)))
    ax.set_yticks(range(len(TRAINABLE_SPECIES)))
    ax.set_xticklabels(TRAINABLE_SPECIES, rotation=75, ha="right", fontsize=8)
    ax.set_yticklabels(TRAINABLE_SPECIES, fontsize=8)
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            ax.text(j, i, str(cm[i, j]), ha="center", va="center", fontsize=7, color="white")
    ax.set_xlabel("predicted")
    ax.set_ylabel("true")
    ax.set_title(f"test confusion  acc={test_acc:.3f}")
    fig.colorbar(im, ax=ax, fraction=0.046)
    fig.tight_layout()
    fig.savefig(path, dpi=120)
    plt.show()


finished = datetime.now(timezone.utc)
record = {
    "kind": "svm",
    "runtime": "colab" if IN_COLAB else "local",
    "started_at": RUN_STARTED.strftime("%Y-%m-%dT%H:%M:%SZ"),
    "elapsed_seconds": round((finished - RUN_STARTED).total_seconds(), 1),
    "data_root": str(DATA_ROOT),
    "max_per_class": MAX_PER_CLASS,
    "split_seed": SPLIT_SEED,
    "image_size": IMAGE_SIZE,
    "variant": TRAIN_VARIANT,
    "feature_spec": FEATURE_SPEC,
    "classes": list(TRAINABLE_SPECIES),
    "held_out": list(HELD_OUT_SPECIES),
    "images_this_run": len(pairs),
    "n_train": len(train_y),
    "n_val": len(val_y),
    "n_test": len(test_y),
    "train_real_counts": real_train_counts,
    "train_class_counts": dict(Counter(train_y)),
    "tuning": TUNING_SPEC,
    "best_params": BEST,
    "best_val_macro_f1": chosen["val_macro_f1"],
    "tuning_rows": tuning_rows,
    "fit_seconds": round(fit_seconds, 1),
    "train_accuracy": results["train"]["accuracy"],
    "val_accuracy": results["val"]["accuracy"],
    "test_accuracy": test_acc,
    "train_report": results["train"]["report"],
    "val_report": results["val"]["report"],
    "test_report": results["test"]["report"],
    "confusion_matrix_test": cm.tolist(),
    "study_target_accuracy": 0.85,
    "loss_note": (
        "RBF SVM is one fit per setting, so there is no epoch train-loss or validation-loss curve. "
        "The tuning curve plots train against validation macro F1 across C instead."
    ),
}
run_dir = append_run(REPO, record)
curve_fig.savefig(run_dir / "tuning_curve.png", dpi=120)
heatmap_fig.savefig(run_dir / "tuning_heatmap.png", dpi=120)
save_f1_chart(run_dir / "f1_scores.png")
save_confusion(run_dir / "confusion_matrix.png")

dest = (
    Path("/content/drive/MyDrive/Seagrass-ml-artifacts/v1")
    if IN_COLAB
    else REPO / "ml" / "artifacts" / "v1"
)
export_artifacts(model, {**record, "run_id": run_dir.name}, dest)
print("wrote", dest)
print("test_accuracy", round(test_acc, 4), "chosen", BEST)
print("history", REPO / "ml" / "artifacts" / "run_log.md")